In [ ]:
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import matplotlib.patches as mpatches

sns.set_theme(palette="muted", style="whitegrid", rc={
    "font.size": 20,
    "axes.titlesize": 14,
    "axes.labelsize": 14,
    "xtick.labelsize": 14,
    "ytick.labelsize": 14,
    "legend.fontsize": 14,
    "legend.title_fontsize": 11,
    "figure.titlesize": 13,
    "axes.spines.right": False,
    "axes.spines.top": False,
})

# Spines must be set after, seaborn's style overrides them
mpl.rcParams["axes.spines.right"] = False
mpl.rcParams["axes.spines.top"] = False

# GEMV

In [ ]:
df = pd.read_csv('csv_output/gemv.csv')
df.head()

In [ ]:
df = pd.read_csv('csv_output/gemv_fsync.csv')
df.head()

In [ ]:
fig, ax = plt.subplots(figsize=(10,6))

for (M, K, N), grp in df.groupby(['M_SIZE', 'K_SIZE', 'N_SIZE']):
    agg = grp.groupby('mesh_dim')['flops_per_cycle'].agg(['mean','std','count']).reset_index()
    agg['stderr'] = agg['std'] / np.sqrt(agg['count'])
    
    ax.errorbar(
        x=agg['mesh_dim'], y=agg['mean'], yerr=agg['stderr'],
        marker='o', capsize=4, label=f'M={M} K={K} N={N}'
    )

xticks = sorted(df['mesh_dim'].unique())
ax.set_xscale('log', base=2)
ax.set_xticks(xticks)
ax.set_xticklabels([f'{x}x{x}' for x in xticks])
ax.set_xlabel('Mesh size')
ax.set_ylabel('Flops/cycle')
ax.set_title('GEMV Scaling')
ax.legend(title='Configs', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

In [ ]:
# sns.set_theme(palette="muted", style="white")
STACK_COLS = ["redmule_cycles", "l2_l1_cycles", "l1_l1_cycles", "fsync_cycles"]
PALETTE = {
    "redmule_cycles":   "#4C72B0",
    "l2_l1_cycles":     "#DD8452",
    "l1_l1_cycles":     "#55A868",
    "fsync_cycles":     "#C44E52",
}
LABELS = {
    "redmule_cycles":   "ReDMulE",
    "l2_l1_cycles":     "L2 → L1",
    "l1_l1_cycles":     "L1 → L1",
    "fsync_cycles":     "Sync",
}

def plot_cycle_breakdown(df: pd.DataFrame, agg: str = "median") -> plt.Figure:
    """
    Stacked bar plot of cycle components, grouped by mesh_dim × (M×K×N) config.
    Parameters
    ----------
    df  : DataFrame with columns: mesh_dim, M_SIZE, K_SIZE, N_SIZE,
          repetition, hartid, and the four cycle columns.
    agg : Aggregation over hartids & repetitions — "median" | "mean" | "sum".
    """
    group_keys = ["mesh_dim", "M_SIZE", "K_SIZE", "N_SIZE"]
    agg_df = df.groupby(group_keys)[STACK_COLS].agg(agg).reset_index()
    agg_df["config"] = (
        agg_df["M_SIZE"].astype(str) + "×"
        + agg_df["K_SIZE"].astype(str) + "×"
        + agg_df["N_SIZE"].astype(str)
    )
    total = agg_df[STACK_COLS].sum(axis=1)
    for col in STACK_COLS:
        agg_df[col + "_pct"] = agg_df[col] / total * 100

    agg_df = agg_df.sort_values(["mesh_dim", "config"]).reset_index(drop=True)
    mesh_dims = sorted(agg_df["mesh_dim"].unique())
    configs   = sorted(agg_df["config"].unique())

    bar_w      = 0.55
    config_gap = bar_w + 0.1
    group_gap  = 1.2

    x_positions   = {}
    group_centers = {}
    group_start   = 0.0
    for md in mesh_dims:
        xs = [group_start + i * config_gap for i, _ in enumerate(configs)]
        for cfg, x in zip(configs, xs):
            x_positions[(md, cfg)] = x
        group_centers[md] = np.mean(xs)
        group_start += len(configs) * config_gap + group_gap

    fig, ax = plt.subplots(figsize=(max(10, len(mesh_dims) * len(configs) * 1.4), 6))

    for _, row in agg_df.iterrows():
        x, bottom = x_positions[(row["mesh_dim"], row["config"])], 0.0
        for col in STACK_COLS:
            val, pct = row[col], row[col + "_pct"]
            ax.bar(x, val, width=bar_w, bottom=bottom,
                   color=PALETTE[col], edgecolor="white", linewidth=0.6, zorder=3)
            if pct >= 5:
                ax.text(x, bottom + val / 2, f"{pct:.1f}%",
                        ha="center", va="center", fontsize=9,
                        color="white", fontweight="bold", zorder=4)
            bottom += val

    bar_xs   = [x_positions[(md, cfg)] for md in mesh_dims for cfg in configs]
    bar_lbls = [cfg for _ in mesh_dims for cfg in configs]

    ax.set_xticks(bar_xs)
    ax.set_xticklabels(bar_lbls, rotation=35, ha="right", color="#444")

    for i, md in enumerate(mesh_dims[:-1]):
        sep_x = (x_positions[(md, configs[-1])] + bar_w / 2
                 + x_positions[(mesh_dims[i + 1], configs[0])] - bar_w / 2) / 2
        ax.axvline(sep_x, color="#bbb", linestyle="--", linewidth=0.8, zorder=1)

    ax.set_ylabel("Cycles")
    ax.set_title("Cycle Breakdown", pad=14)
    ax.yaxis.grid(True, linestyle="--", alpha=0.5, zorder=0)
    ax.set_axisbelow(True)
    ax.spines[["top", "right"]].set_visible(False)
    ax.set_xlim(bar_xs[0] - bar_w, bar_xs[-1] + bar_w)

    ax.legend(
        handles=[mpatches.Patch(color=PALETTE[c], label=LABELS[c]) for c in STACK_COLS],
        loc="upper right", framealpha=0.9, title="Component"
    )

    ax2 = ax.twiny()
    ax2.set_xlim(ax.get_xlim())
    ax2.set_xticks([group_centers[md] for md in mesh_dims])
    ax2.set_xticklabels([f"{md}×{md}" for md in mesh_dims], fontweight="bold", color="#222")
    ax2.spines[["bottom", "right", "left"]].set_visible(False)
    ax2.tick_params(axis="x", length=0)

    fig.tight_layout()
    return fig

fig = plot_cycle_breakdown(df, agg="median")

In [ ]:
STACK_COLS = ["redmule_cycles", "l2_l1_cycles", "l1_l1_cycles", "fsync_cycles"]
LABELS = {
    "redmule_cycles":   "ReDMulE",
    "l2_l1_cycles":     "L2 → L1",
    "l1_l1_cycles":     "L1 → L1",
    "fsync_cycles": "Sync",
}
MESH_DIMS = [2, 4, 8, 16]   # expected columns; missing ones are left blank
CMAPS = {
    "redmule_cycles":   "Blues",
    "l2_l1_cycles":     "Oranges",
    "l1_l1_cycles":     "Greens",
    "fsync_cycles": "Reds",
}
 
 
def plot_heatmap_grid(df: pd.DataFrame, M: int, K: int, N: int) -> plt.Figure:
    """
    One figure for a single (M, K, N) configuration.
    Layout: 4 rows (cycle components) × 4 columns (mesh_dim 2,4,8,16).
    Each cell is a heatmap of y_id (row) vs x_id (col), value = median
    cycles across repetitions.
 
    Parameters
    ----------
    df      : Full DataFrame (all mesh_dims, all configs).
    M, K, N : The configuration to plot.
    """
    sub = df[(df["M_SIZE"] == M) & (df["K_SIZE"] == K) & (df["N_SIZE"] == N)].copy()
    if sub.empty:
        raise ValueError(f"No data found for M={M} K={K} N={N}")
 
    # Derive spatial ids
    sub["x_id"] = sub["hartid"] % sub["mesh_dim"]
    sub["y_id"] = sub["hartid"] // sub["mesh_dim"]
 
    n_rows = len(STACK_COLS)
    n_cols = len(MESH_DIMS)
 
    fig, axes = plt.subplots(
        n_rows, n_cols,
        figsize=(2.5 * n_cols, 2.0 * n_rows),
        squeeze=False,
    )
 
    fig.suptitle(
        f"Cycle Heatmaps  —  M={M}, K={K}, N={N}",
        fontweight="bold", y=1.01,
    )
 
    for col_idx, md in enumerate(MESH_DIMS):
        md_data = sub[sub["mesh_dim"] == md]
 
        for row_idx, cyc_col in enumerate(STACK_COLS):
            ax = axes[row_idx][col_idx]
 
            if md_data.empty:
                ax.set_visible(False)
                continue
 
            # Aggregate: median across repetitions per (y_id, x_id)
            pivot = (
                md_data
                .groupby(["y_id", "x_id"])[cyc_col]
                .median()
                .unstack(level="x_id")   # columns = x_id
                .sort_index(ascending=False)   # y_id=0 at top looks natural flipped; keep ascending=True for row-major top-down
            )
            # Sort so y_id=0 is at the top (row-major, top-down)
            pivot = pivot.sort_index(ascending=True)
 
            # Ensure full grid even if some hartids are missing
            full_idx = range(md)
            pivot = pivot.reindex(index=full_idx, columns=full_idx)
 
            sns.heatmap(
                pivot,
                ax=ax,
                cmap=CMAPS[cyc_col],
                annot=(md <= 8),          # annotate cells only for small grids
                fmt=".0f",
                annot_kws={"fontsize": 0},
                linewidths=0.4,
                linecolor="#ddd",
                cbar=True,
                cbar_kws={"shrink": 0.75, "pad": 0.02},
            )
 
            # Titles / labels
            if row_idx == 0:
                ax.set_title(f"mesh {md}×{md}", fontweight="bold", pad=6)
            if col_idx == 0:
                ax.set_ylabel(LABELS[cyc_col], labelpad=6)
            else:
                ax.set_ylabel("")
 
            ax.set_xlabel("x_id" if row_idx == n_rows - 1 else "")
            ax.tick_params(axis="both", labelsize=7)
            ax.set_yticklabels(ax.get_yticklabels(), rotation=0)
            ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
 
    fig.tight_layout()
    return fig
 
 
def plot_all_configs(df: pd.DataFrame) -> dict:
    """
    Convenience wrapper: iterates over every unique (M, K, N) in df
    and returns a dict {(M, K, N): fig}.
    """
    configs = df[["M_SIZE", "K_SIZE", "N_SIZE"]].drop_duplicates().values
    return {
        (M, K, N): plot_heatmap_grid(df, M, K, N)
        for M, K, N in configs
    }
figs = plot_all_configs(df)

# MM Input-Stationary

# MM Weight-Stationary

# MM Output-Stationary